# Chia CSV lớn thành các chunk

Notebook chia `data/dark.csv` sang `data/chunks/` theo cơ chế ghi file tạm rồi đổi tên, tránh để lại chunk hỏng khi tiến trình bị ngắt.

## 1. Cấu hình

Thao tác chỉ chạy khi đặt `RUN_DATA_SPLIT=1`; mặc định không xóa bất kỳ file nguồn nào.

In [1]:
from pathlib import Path
import os

import pandas as pd

DATA_DIR = Path('data')
SOURCE_FILE = DATA_DIR / 'dark.csv'
CHUNK_DIR = DATA_DIR / 'chunks'
ROWS_PER_FILE = int(os.getenv('ROWS_PER_FILE', '15000000'))
RUN_SPLIT = os.getenv('RUN_DATA_SPLIT') == '1'
DELETE_DERIVED_SOURCES = os.getenv('DELETE_DERIVED_SOURCES') == '1'

{
    'source': str(SOURCE_FILE),
    'source_exists': SOURCE_FILE.exists(),
    'rows_per_file': ROWS_PER_FILE,
    'output_dir': str(CHUNK_DIR),
}

{'source': 'data\\dark.csv',
 'source_exists': False,
 'rows_per_file': 15000000,
 'output_dir': 'data\\chunks'}

## 2. Hàm chia an toàn

Các file cũ trong `data/chunks/` được thay thế có chủ đích; CSV ở thư mục `data/` không bị đụng tới.

In [2]:
def split_csv(source: Path, output_dir: Path, rows_per_file: int) -> list[Path]:
    if not source.exists():
        raise FileNotFoundError(
            f'{source} chưa tồn tại; hãy chạy merge.ipynb trước.'
        )
    output_dir.mkdir(parents=True, exist_ok=True)
    written = []
    for index, frame in enumerate(pd.read_csv(source, chunksize=rows_per_file), start=1):
        destination = output_dir / f'dark_{index}.csv'
        temporary = destination.with_suffix('.csv.tmp')
        frame.to_csv(temporary, index=False)
        temporary.replace(destination)
        written.append(destination)
        print(f'Đã ghi {destination} ({len(frame):,} dòng)')
    return written

In [3]:
if RUN_SPLIT:
    chunks = split_csv(SOURCE_FILE, CHUNK_DIR, ROWS_PER_FILE)
    if DELETE_DERIVED_SOURCES:
        for name in ('dark.csv', 'light.csv', 'animal_chess.csv'):
            (DATA_DIR / name).unlink(missing_ok=True)
        print('Đã xóa các file dẫn xuất lớn sau khi chia thành công.')
    print(f'Hoàn tất {len(chunks)} chunk.')
else:
    print('Chế độ preview: đặt RUN_DATA_SPLIT=1 để bắt đầu chia file.')

Chế độ preview: đặt RUN_DATA_SPLIT=1 để bắt đầu chia file.


## Tóm tắt

- Chunk được lưu riêng trong `data/chunks/`, không ghi đè ba file LFS gốc.
- Ghi qua `.tmp` rồi đổi tên để hạn chế file dở dang.
- Xóa file dẫn xuất là tùy chọn và chỉ xảy ra sau khi chia thành công.